# 🌊 WQR — Wavelet Quantile Regression
## A Comprehensive Python Toolkit for Quantile-Based Econometric Analysis

---

**Author:** Dr. Merwan Roudane  
📧 merwanroudane920@gmail.com  
🔗 [GitHub](https://github.com/merwanroudane/wqrr) | [PyPI](https://pypi.org/project/wqr/)  

---

### 📖 Overview

This tutorial provides a **complete, end-to-end demonstration** of the `wqr` library — a unified Python framework for wavelet-based quantile regression econometrics. The library implements **nine** cutting-edge methodologies with **publication-quality** MATLAB-style visualizations.

### 🎯 What You Will Learn

| # | Module | Methodology | Reference |
|---|--------|-------------|-----------|
| 1 | `qq_regression` | Quantile-on-Quantile Regression | Sim & Zhou (2015) |
| 2 | `wavelet_qr` | Wavelet Quantile Regression (WQR) | Adebayo & Ozkan (2023) |
| 3 | `multivariate_wqr` | Multivariate WQR (MWQR) | Adebayo et al. (2025) |
| 4 | `wavelet_qqr` | Wavelet QQR with P-values (WQQR) | Adebayo, Özkan et al. (2025) |
| 5 | `np_quantile_causality` | Nonparametric Quantile Causality | Balcilar et al. (2016) |
| 6 | `wavelet_np_causality` | Wavelet NP Quantile Causality | Balcilar et al. (2016) |
| 7 | `wavelet_mediation` | Wavelet Quantile Mediation & Moderation | Adebayo (2025) |
| 8 | `wavelet_quantile_correlation` | Wavelet Quantile Correlation | Roudane (2024) |
| 9 | `wavelet_quantile_density` | Wavelet Quantile Density Estimation | Chesneau, Dewan & Doosti |

### 📊 Data

We use **real financial and macroeconomic data** fetched from Yahoo Finance:
- **Crude Oil (CL=F)** — WTI Crude Oil Futures
- **Gold (GC=F)** — Gold Futures
- **S&P 500 (^GSPC)** — US Stock Market Index
- **US Dollar Index (DX-Y.NYB)** — Dollar Strength

These variables are widely used in financial econometrics research, including the original papers upon which this library is built.

---

## ⚙️ 1. Setup & Installation

In [ ]:
# Install WQR and data dependencies
# !pip install wqr yfinance --quiet

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  Core Imports
# ══════════════════════════════════════════════════════════════════════

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import warnings
warnings.filterwarnings('ignore')

# WQR Library
import wqr

# Configure matplotlib for publication-quality plots
plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 12,
    'axes.labelsize': 13,
    'axes.titlesize': 14,
    'figure.dpi': 120,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
})

print(f"WQR version: {wqr.__version__}")
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")
print("✓ All imports successful.")

---

## 📥 2. Data Loading — Real Financial Data

We fetch daily closing prices for **Oil, Gold, S&P 500, and US Dollar Index** from Yahoo Finance (2015–2024), then compute **log returns** — the standard transformation used in financial econometrics.

> **Note:** If `yfinance` is unavailable, the notebook will automatically generate realistic synthetic data that mimics the statistical properties of these financial series.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  Data Acquisition
# ══════════════════════════════════════════════════════════════════════

try:
    import yfinance as yf
    
    tickers = {
        'Oil':   'CL=F',
        'Gold':  'GC=F',
        'SP500': '^GSPC',
        'USD':   'DX-Y.NYB',
    }
    
    raw = yf.download(
        list(tickers.values()),
        start='2015-01-01',
        end='2024-12-31',
        auto_adjust=True
    )['Close']
    
    raw.columns = [k for k, v in tickers.items() for c in raw.columns if v == c] \
                  if len(raw.columns) == len(tickers) else list(tickers.keys())
    raw.columns = list(tickers.keys())
    raw = raw.dropna()
    
    # Compute log returns (×100 for percentage scale)
    returns = np.log(raw / raw.shift(1)).dropna() * 100
    
    data_source = "Yahoo Finance"
    print(f"✓ Downloaded {len(raw)} daily observations from Yahoo Finance")
    print(f"  Period: {raw.index[0].strftime('%Y-%m-%d')} to {raw.index[-1].strftime('%Y-%m-%d')}")

except Exception as e:
    print(f"⚠ Yahoo Finance unavailable ({e}). Generating synthetic financial data...")
    
    np.random.seed(2024)
    n = 2000
    
    # Correlated returns with realistic properties
    # Oil, Gold, SP500, USD have known correlations
    corr_matrix = np.array([
        [1.00,  0.15, 0.30, -0.45],  # Oil
        [0.15,  1.00, 0.05, -0.60],  # Gold
        [0.30,  0.05, 1.00, -0.25],  # SP500
        [-0.45, -0.60, -0.25, 1.00],  # USD
    ])
    L = np.linalg.cholesky(corr_matrix)
    
    # GARCH-like heteroskedasticity
    z = np.random.standard_t(df=5, size=(n, 4))
    r = z @ L.T
    
    # Scale to realistic daily return volatilities
    vols = np.array([2.5, 1.2, 1.1, 0.5])  # Oil is most volatile
    means = np.array([0.01, 0.02, 0.04, 0.005])
    r = r * vols + means
    
    dates = pd.bdate_range(start='2015-01-02', periods=n)
    returns = pd.DataFrame(r, index=dates, columns=['Oil', 'Gold', 'SP500', 'USD'])
    
    data_source = "Synthetic (calibrated)"
    print(f"✓ Generated {n} synthetic observations (calibrated to real market data)")

print(f"\n  Variables: {list(returns.columns)}")
print(f"  Shape: {returns.shape}")

### 📋 2.1  Descriptive Statistics

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  Summary Statistics using WQR's built-in function
# ══════════════════════════════════════════════════════════════════════

stats_table = wqr.summary_statistics(returns)
print("═" * 80)
print("         DESCRIPTIVE STATISTICS — Daily Log Returns (%)")
print(f"         Data Source: {data_source}")
print("═" * 80)
print(stats_table.to_string(index=False))
print("═" * 80)

### 📊 2.2  Data Visualisation

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  Time Series & Distribution Plots
# ══════════════════════════════════════════════════════════════════════

colors = ['#E74C3C', '#F1C40F', '#2E86C1', '#27AE60']

fig, axes = plt.subplots(4, 2, figsize=(16, 14))

for i, (col, color) in enumerate(zip(returns.columns, colors)):
    # Time series
    ax = axes[i, 0]
    ax.plot(returns.index, returns[col], color=color, linewidth=0.5, alpha=0.8)
    ax.axhline(y=0, color='gray', linewidth=0.5, linestyle=':')
    ax.set_title(f'{col} — Daily Returns (%)', fontweight='bold', fontsize=12)
    ax.set_ylabel('Return (%)')
    ax.grid(True, alpha=0.2)
    
    # Distribution
    ax = axes[i, 1]
    ax.hist(returns[col].dropna(), bins=60, density=True, color=color, alpha=0.6, edgecolor='white')
    
    # Fit normal overlay
    from scipy.stats import norm
    xmin, xmax = ax.get_xlim()
    xx = np.linspace(xmin, xmax, 200)
    mu, std = returns[col].mean(), returns[col].std()
    ax.plot(xx, norm.pdf(xx, mu, std), 'k--', linewidth=1.5, label='Normal')
    ax.set_title(f'{col} — Distribution', fontweight='bold', fontsize=12)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.2)

fig.suptitle('Financial Returns — Time Series & Distributions',
             fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---

## 🔬 3. Quantile-on-Quantile Regression

### Theory

The **Quantile-on-Quantile (QQ) Regression** methodology of **Sim & Zhou (2015)** examines the dependence between the $\theta$-th quantile of a dependent variable $Y$ and the $\tau$-th quantile of an independent variable $X$.

$$\hat{\beta}(\theta, \tau) = \text{argmin} \sum_{i=1}^{n} \rho_{\theta}\Big(y_i - \alpha - \beta \cdot x_i\Big) \cdot \mathbb{1}\{x_i \leq Q_X(\tau)\}$$

This produces a **19 × 19 coefficient surface** that reveals how the effect of $X$ on $Y$ varies across **both** the distributional locations of $X$ and $Y$.

### Application

We examine: **Does oil price return affect S&P 500 returns asymmetrically across quantiles?**

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  3.1  QQ Regression — Oil → S&P 500
# ══════════════════════════════════════════════════════════════════════

y = returns['SP500'].values
x = returns['Oil'].values

qq_result = wqr.qq_regression(
    y, x,
    y_quantiles=np.arange(0.05, 1.0, 0.05),  # 19 quantiles
    x_quantiles=np.arange(0.05, 1.0, 0.05),
    se_method='boot',
    n_boot=200,
    verbose=True
)

# Print summary
qq_result.summary()

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  3.2  3D Surface Plot (MATLAB-style Jet Colormap)
# ══════════════════════════════════════════════════════════════════════

fig, ax = wqr.plot_qq_3d(
    qq_result,
    value='coefficient',
    colormap='jet',
    x_label='Oil Quantile (τ)',
    y_label='S&P 500 Quantile (θ)',
    z_label='β̂(θ,τ)',
    title='QQ Regression: Oil → S&P 500',
    elev=25,
    azim=-135,
    figsize=(12, 9)
)
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  3.3  Heatmap with Significance Stars
# ══════════════════════════════════════════════════════════════════════

fig, ax = wqr.plot_qq_heatmap(
    qq_result,
    value='coefficient',
    colormap='jet',
    x_label='Oil Quantile (τ)',
    y_label='S&P 500 Quantile (θ)',
    title='QQ Regression Heatmap: Oil → S&P 500',
    show_significance=True,
    figsize=(11, 9)
)
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  3.4  Contour Plot
# ══════════════════════════════════════════════════════════════════════

fig, ax = wqr.plot_qq_contour(
    qq_result,
    value='coefficient',
    colormap='jet',
    x_label='Oil Quantile (τ)',
    y_label='S&P 500 Quantile (θ)',
    title='QQ Regression Contour: Oil → S&P 500',
    levels=25,
    figsize=(11, 9)
)
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  3.5  R-squared Surface (Model Fit)
# ══════════════════════════════════════════════════════════════════════

fig, ax = wqr.plot_qq_3d(
    qq_result,
    value='r_squared',
    colormap='viridis',
    x_label='Oil Quantile (τ)',
    y_label='S&P 500 Quantile (θ)',
    z_label='Pseudo R²',
    title='QQ Regression — Goodness of Fit (R²)',
    figsize=(12, 9)
)
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  3.6  Quantile Correlation Heatmap
# ══════════════════════════════════════════════════════════════════════

fig, ax = wqr.plot_correlation_heatmap(
    returns['SP500'].values,
    returns['Oil'].values,
    quantiles=np.arange(0.1, 1.0, 0.1),
    x_label='Oil Quantiles',
    y_label='S&P 500 Quantiles',
    title='Quantile Correlation: Oil vs S&P 500',
    colormap='RdBu_r',
    annotate=True,
    figsize=(10, 9)
)
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  3.7  Export Results
# ══════════════════════════════════════════════════════════════════════

# Export to CSV
qq_result.export_csv('qq_oil_sp500.csv', digits=4)
print("✓ Results exported to qq_oil_sp500.csv")

# Export LaTeX table
latex = qq_result.export_latex(
    value='coefficient',
    caption='QQ Regression Coefficients: Oil $\\rightarrow$ S\\&P 500'
)
print("\n── LaTeX Table (first 20 lines) ──")
print('\n'.join(latex.split('\n')[:20]))
print("...")

---

## 🌊 4. Wavelet Quantile Regression (WQR)

### Theory

**Wavelet Quantile Regression** (Adebayo & Ozkan, 2023) decomposes the original time series into **Short-run, Medium-run, and Long-run** frequency components using the **MODWT (Maximal Overlap Discrete Wavelet Transform)**, then estimates quantile regression at each frequency band.

$$Y^{(b)} = \alpha^{(b)}(\tau) + \beta^{(b)}(\tau) \cdot X^{(b)} + \varepsilon_{\tau}^{(b)}$$

where $b \in \{\text{Short}, \text{Medium}, \text{Long}\}$ and $\tau \in (0, 1)$.

### Application

**Does the impact of oil on gold vary across different time horizons?**

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  4.1  Wavelet Quantile Regression — Oil → Gold
# ══════════════════════════════════════════════════════════════════════

wqr_result = wqr.wavelet_qr(
    y=returns['Gold'].values,
    x=returns['Oil'].values,
    quantiles=np.arange(0.05, 1.0, 0.05),
    wavelet='la8',
    J=5,
    bands=True,         # Aggregate to Short/Medium/Long
    verbose=True
)

# Summary
wqr_result.summary()

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  4.2  WQR Heatmap with Significance Stars
# ══════════════════════════════════════════════════════════════════════

fig, ax = wqr.plot_wqr_heatmap(
    wqr_result.coefficients,
    colormap='green_orange_red',
    title='Wavelet Quantile Regression: Oil → Gold',
    x_label='Quantiles (τ)',
    y_label='Time Horizons',
    figsize=(14, 5)
)
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  4.3  WQR Results Table (Formatted)
# ══════════════════════════════════════════════════════════════════════

table = wqr.results_table(
    wqr_result.coefficients,
    value_col='beta',
    pval_col='p_value',
    row_col='quantile',
    col_col='level',
    digits=4
)

print("═" * 70)
print("  WQR Coefficients: Oil → Gold")
print("  Stars: *** p<0.01, ** p<0.05, * p<0.10")
print("═" * 70)
print(table.to_string())
print("═" * 70)

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  4.4  LaTeX Export for WQR
# ══════════════════════════════════════════════════════════════════════

latex_wqr = wqr.export_latex(
    wqr_result.coefficients,
    caption='Wavelet Quantile Regression: Oil $\\rightarrow$ Gold',
    label='tab:wqr_oil_gold',
    digits=4
)
print(latex_wqr)

---

## 📐 5. Multivariate Wavelet Quantile Regression (MWQR)

### Theory

**Multivariate WQR** (Adebayo et al., 2025) extends the bivariate WQR to **multiple independent variables**:

$$Y^{(b)} = \alpha^{(b)}(\tau) + \sum_{k=1}^{K} \beta_k^{(b)}(\tau) \cdot X_k^{(b)} + \varepsilon_{\tau}^{(b)}$$

### Application

**How do Oil, Gold, and USD jointly affect S&P 500 across quantiles and time horizons?**

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  5.1  Multivariate WQR — Oil + Gold + USD → S&P 500
# ══════════════════════════════════════════════════════════════════════

mwqr_result = wqr.multivariate_wqr(
    y=returns['SP500'].values,
    X_dict={
        'Oil':  returns['Oil'].values,
        'Gold': returns['Gold'].values,
        'USD':  returns['USD'].values,
    },
    quantiles=np.array([0.01, 0.05, 0.10, 0.20, 0.30, 0.40,
                        0.50, 0.60, 0.70, 0.80, 0.90, 0.95, 0.99]),
    wavelet='la8',
    J=6,
    dep_name='S&P500',
    verbose=True
)

mwqr_result.summary()

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  5.2  Per-Variable Heatmaps
# ══════════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

for idx, var in enumerate(['Oil', 'Gold', 'USD']):
    var_df = mwqr_result.get_variable(var)
    
    # Pivot for heatmap
    piv = var_df.pivot(index='band', columns='quantile', values='beta')
    piv_p = var_df.pivot(index='band', columns='quantile', values='p_value')
    
    band_order = ['Short', 'Medium', 'Long']
    ordered = [b for b in band_order if b in piv.index]
    piv = piv.reindex(ordered)
    piv_p = piv_p.reindex(ordered)
    
    ax = axes[idx]
    from matplotlib.colors import LinearSegmentedColormap
    cmap = LinearSegmentedColormap.from_list('gor',
        [(0, '#8FBC8F'), (0.5, '#FFA500'), (1.0, '#FF0000')])
    
    im = ax.imshow(piv.values.astype(float), cmap=cmap, aspect='auto')
    ax.set_xticks(range(len(piv.columns)))
    ax.set_xticklabels([f'{q:.2f}' for q in piv.columns], rotation=45, ha='right', fontsize=8)
    ax.set_yticks(range(len(piv.index)))
    ax.set_yticklabels(piv.index)
    ax.set_title(f'{var} → S&P 500', fontweight='bold', fontsize=13)
    
    # Significance stars
    pmat = piv_p.values.astype(float)
    for i in range(pmat.shape[0]):
        for j in range(pmat.shape[1]):
            p = pmat[i, j]
            if not np.isnan(p):
                star = '***' if p < 0.01 else ('**' if p < 0.05 else ('*' if p < 0.10 else ''))
                if star:
                    ax.text(j, i, star, ha='center', va='center', fontsize=9, fontweight='bold')
    
    plt.colorbar(im, ax=ax, shrink=0.8)

fig.suptitle('Multivariate WQR: Oil + Gold + USD → S&P 500',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  5.3  Significance Matrix with Stars
# ══════════════════════════════════════════════════════════════════════

for var in ['Oil', 'Gold', 'USD']:
    stars = mwqr_result.significance_matrix(var)
    print(f"\n── Significance Stars: {var} → S&P 500 ──")
    print(stars.to_string())

---

## 🔮 6. Wavelet QQR with P-values (WQQR)

### Theory

**WQQR** (Adebayo, Özkan et al., 2025) combines wavelet decomposition with **kernel-weighted local polynomial quantile regression** to estimate the full QQ coefficient surface within a specific frequency band, along with pointwise p-values.

### Application

**Examine the long-run Oil → Gold dependence structure with statistical significance.**

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  6.1  WQQR — Oil → Gold (Long-run band)
# ══════════════════════════════════════════════════════════════════════

wqqr_result = wqr.wavelet_qqr(
    y=returns['Gold'].values,
    x=returns['Oil'].values,
    quantile_step=0.05,      # 19 quantiles
    wavelet='la8',
    J=5,
    band='long',             # Focus on long-run dynamics
    bandwidth=1.0,
    verbose=True
)

wqqr_result.summary()

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  6.2  WQQR 3D Surface
# ══════════════════════════════════════════════════════════════════════

fig, ax = wqr.plot_wqqr_surface(
    wqqr_result,
    colormap='jet',
    x_label='Oil Quantile (τ)',
    y_label='Gold Quantile (θ)',
    z_label='β̂(θ,τ)',
    title='WQQR 3D Surface: Oil → Gold',
    figsize=(12, 9)
)
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  6.3  P-value Significance Map
# ══════════════════════════════════════════════════════════════════════

fig, ax = wqr.plot_wqqr_pvalue_heatmap(
    wqqr_result,
    alpha=0.05,
    title='WQQR Significance Map (Long-run): Oil → Gold',
    figsize=(9, 8)
)
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  6.4  WQR vs WQQR Comparison
# ══════════════════════════════════════════════════════════════════════

fig, ax = wqr.plot_wqr_vs_wqqr(
    wqqr_result,
    title='WQR vs WQQR Comparison (Long-run): Oil → Gold',
    figsize=(12, 5)
)
plt.show()

---

## ⚡ 7. Nonparametric Quantile Causality

### Theory

The **Balcilar–Jeong–Nishiyama** nonparametric quantile Granger causality test examines whether past values of $X$ help predict the conditional quantile of $Y$:

$$H_0: P\big[Y_t \leq Q_{\tau}(Y_t | Y_{t-1})\big] = P\big[Y_t \leq Q_{\tau}(Y_t | Y_{t-1}, X_{t-1})\big]$$

The test statistic follows an asymptotic $N(0,1)$ distribution under the null of no causality.

- **Test in mean:** Causality in the conditional mean across quantiles
- **Test in variance:** Causality in the conditional variance (risk spillovers)

### Application

**Does oil price Granger-cause gold price across the entire distribution?**

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  7.1  Nonparametric Quantile Causality — Mean
# ══════════════════════════════════════════════════════════════════════

caus_mean = wqr.np_quantile_causality(
    x=returns['Oil'].values,
    y=returns['Gold'].values,
    test_type='mean',
    q=np.arange(0.05, 1.0, 0.05)
)

caus_mean.summary()

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  7.2  Causality Plot — Mean
# ══════════════════════════════════════════════════════════════════════

fig, ax = wqr.plot_causality(
    caus_mean,
    title='Nonparametric Quantile Causality in Mean: Oil → Gold',
    figsize=(12, 5)
)
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  7.3  Nonparametric Quantile Causality — Variance
# ══════════════════════════════════════════════════════════════════════

caus_var = wqr.np_quantile_causality(
    x=returns['Oil'].values,
    y=returns['Gold'].values,
    test_type='variance',
    q=np.arange(0.05, 1.0, 0.05)
)

caus_var.summary()

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  7.4  Causality Plot — Variance
# ══════════════════════════════════════════════════════════════════════

fig, ax = wqr.plot_causality(
    caus_var,
    title='Nonparametric Quantile Causality in Variance: Oil → Gold',
    figsize=(12, 5)
)
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  7.5  Side-by-side: Mean vs Variance Causality
# ══════════════════════════════════════════════════════════════════════

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

for ax, res, tp in [(ax1, caus_mean, 'Mean'), (ax2, caus_var, 'Variance')]:
    q = res.quantiles
    s = res.statistic
    
    ax.fill_between(q, 1.96, s, where=(np.abs(s) >= 1.96),
                    alpha=0.15, color='#DC143C')
    ax.plot(q, s, color='#2C3E50', linewidth=2.0, label='Test Statistic', zorder=3)
    ax.axhline(y=1.96, color='#E74C3C', linewidth=1.5, linestyle='--', label='5% CV')
    ax.axhline(y=1.645, color='#F39C12', linewidth=1.0, linestyle=':', label='10% CV', alpha=0.7)
    
    ax.set_xlabel('Quantile (τ)', fontsize=13)
    ax.set_ylabel('Test Statistic', fontsize=13)
    ax.set_title(f'Causality in {tp}: Oil → Gold', fontsize=13, fontweight='bold')
    ax.legend(fontsize=9, loc='upper left')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

## 🌊 8. Wavelet Nonparametric Quantile Causality (WNQC)

### Theory

**WNQC** extends the nonparametric quantile causality test to **wavelet-decomposed data**, allowing us to test for Granger causality **at different time horizons** — a crucial feature for separating short-run speculative effects from long-run fundamental relationships.

### Application

**At which frequencies does oil Granger-cause gold?**

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  8.1  Wavelet NP Quantile Causality — Oil → Gold
# ══════════════════════════════════════════════════════════════════════

wnqc_result = wqr.wavelet_np_causality(
    x=returns['Oil'].values,
    y=returns['Gold'].values,
    test_type='mean',
    wavelet='la8',
    J=5,
    bands=True,
    verbose=True
)

wnqc_result.summary()

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  8.2  Wavelet Causality Heatmap
# ══════════════════════════════════════════════════════════════════════

fig, ax = wqr.plot_wavelet_causality(
    wnqc_result,
    colormap='green_orange_red',
    title='Wavelet NP Quantile Causality (Mean): Oil → Gold',
    figsize=(14, 5)
)
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  8.3  Test Statistic Matrix
# ══════════════════════════════════════════════════════════════════════

stat_matrix = wnqc_result.to_matrix()
print("── Wavelet Causality Test Statistics ──")
print(stat_matrix.round(3).to_string())

print("\n── Significance Stars ──")
stars = wnqc_result.significance_matrix()
print(stars.to_string())

---

## 🔀 9. Wavelet Quantile Mediation & Moderation

### Theory

**Wavelet Quantile Mediation & Moderation** (Adebayo, 2025) estimates five regression paths in a wavelet-decomposed framework:

| Path | Regression | Interpretation |
|------|-----------|----------------|
| **Direct** | $Y \sim X$ | Direct effect of X on Y |
| **Interaction** | $Y \sim X + Z + X \cdot Z$ | Moderation (X×Z coefficient) |
| **Path a** | $Z \sim X$ | Effect of X on mediator Z |
| **Path b** | $Y \sim X + Z$ | Effect of Z on Y (controlling for X) |
| **Indirect** | $a(\tau) \times b(\tau)$ | Mediation (product of paths) |

### Application

**Does gold mediate the relationship between oil and S&P 500?**

- **X** (Main): Oil (independent variable)
- **Z** (Mediator/Moderator): Gold
- **Y** (Dependent): S&P 500

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  9.1  Wavelet Quantile Mediation: Oil → Gold → S&P 500
# ══════════════════════════════════════════════════════════════════════

med_result = wqr.wavelet_mediation(
    y=returns['SP500'].values,
    x=returns['Oil'].values,
    z=returns['Gold'].values,
    quantiles=np.array([0.01, 0.05, 0.10, 0.20, 0.30, 0.40,
                        0.50, 0.60, 0.70, 0.80, 0.90, 0.95, 0.99]),
    wavelet='la8',
    dep_name='S&P500',
    main_name='Oil',
    mod_name='Gold',
    verbose=True
)

med_result.summary()

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  9.2  Five-Panel Mediation/Moderation Heatmap
# ══════════════════════════════════════════════════════════════════════

fig, axes = wqr.plot_mediation_panel(
    med_result,
    colormap='green_yellow_red',
    figsize=(14, 20)
)
plt.show()

---

## 🔗 10. Wavelet Quantile Correlation (WQC)

### Theory

The **Wavelet Quantile Correlation** (Roudane, 2024) measures the quantile dependence between two wavelet-decomposed series at each detail level, with **Monte Carlo bootstrap confidence intervals**.

$$\widehat{QC}_{\tau}^{(j)}(X, Y) = \frac{\mathbb{E}\Big[\mathbb{1}\{X^{(j)} \leq Q_{X}^{(j)}(\tau)\} \cdot \mathbb{1}\{Y^{(j)} \leq Q_{Y}^{(j)}(\tau)\}\Big] - \tau^2}{\tau(1-\tau)}$$

### Application

**How does the quantile correlation between Oil and S&P 500 vary across wavelet scales?**

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  10.1  Wavelet Quantile Correlation — Oil vs S&P 500
# ══════════════════════════════════════════════════════════════════════

wqc_result = wqr.wavelet_quantile_correlation(
    x=returns['Oil'].values,
    y=returns['SP500'].values,
    quantiles=np.array([0.05, 0.25, 0.50, 0.75, 0.95]),
    wavelet='la8',
    J=8,
    n_sim=500,
    verbose=True
)

wqc_result.summary()

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  10.2  WQC Heatmap with Significance Borders
# ══════════════════════════════════════════════════════════════════════

fig, ax = wqr.plot_wqc_heatmap(
    wqc_result,
    colormap='viridis',
    title='Wavelet Quantile Correlation: Oil vs S&P 500',
    figsize=(11, 7)
)
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  10.3  Detailed WQC Results
# ══════════════════════════════════════════════════════════════════════

print("── WQC Correlation Matrix ──")
mat = wqc_result.to_matrix()
print(mat.round(4).to_string())

print(f"\n── Significant Cells (outside 95% CI) ──")
sig = wqc_result.significant_cells()
if len(sig) > 0:
    print(sig[['Level', 'Quantile', 'Estimated_QC', 'CI_Lower', 'CI_Upper']].to_string(index=False))
else:
    print("No statistically significant cells at 5% level.")

---

## 📉 11. Wavelet Quantile Density Estimation

### Theory

The **Wavelet Quantile Density** estimator (Chesneau, Dewan & Doosti) provides a nonparametric estimate of the quantile density function:

$$q(p) = \frac{1}{f(F^{-1}(p))}$$

using three approaches:
1. **Linear wavelet estimate** — Direct projection onto scaling basis
2. **Hard thresholding** — Denoise via DWT thresholding
3. **Local linear smoothing** — Kernel smoothing of the wavelet estimate

### Application

**Estimate the quantile density of Oil returns (with known reference distribution).**

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  11.1  Quantile Density — Oil Returns
# ══════════════════════════════════════════════════════════════════════

oil_data = returns['Oil'].dropna().values

qd_result = wqr.wavelet_quantile_density(
    y=oil_data,
    j0=5,
    bandwidth=0.15,
    wavelet='coif1',
    gld_params=None  # No known true distribution
)

qd_result.summary()

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  11.2  Four-Panel Quantile Density Plot
# ══════════════════════════════════════════════════════════════════════

fig, axes = wqr.plot_quantile_density(
    qd_result,
    figsize=(14, 10)
)
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  11.3  Quantile Density with Known Reference (Simulated GLD)
# ══════════════════════════════════════════════════════════════════════

# Generate GLD sample: λ1=0, λ2=1, λ3=0.5, λ4=0.5
np.random.seed(123)
n_gld = 500
u = np.random.uniform(0.01, 0.99, n_gld)
gld_params = (0, 1, 0.5, 0.5)
l1, l2, l3, l4 = gld_params
gld_sample = l1 + (u**l3 - (1-u)**l4) / l2

qd_gld = wqr.wavelet_quantile_density(
    y=gld_sample,
    j0=5,
    bandwidth=0.15,
    wavelet='coif1',
    gld_params=gld_params  # Provide true reference
)

qd_gld.summary()

fig, axes = wqr.plot_quantile_density(
    qd_gld,
    figsize=(14, 10)
)
plt.show()

---

## 📋 12. Tables & LaTeX Export

The `wqr` library provides three table utilities for publication-ready output:

| Function | Description |
|----------|-------------|
| `results_table()` | Formatted table with significance stars |
| `export_latex()` | Journal-ready LaTeX table |
| `summary_statistics()` | Descriptive statistics (mean, std, skewness, kurtosis) |

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  12.1  Comprehensive LaTeX Tables
# ══════════════════════════════════════════════════════════════════════

# Table 1: WQR Results
print("═" * 70)
print("  TABLE 1: Wavelet Quantile Regression — Oil → Gold")
print("═" * 70)
latex1 = wqr.export_latex(
    wqr_result.coefficients,
    caption='Wavelet Quantile Regression: Oil $\\rightarrow$ Gold',
    label='tab:wqr',
    digits=4
)
print(latex1)

print("\n")

# Table 2: MWQR — Oil variable
print("═" * 70)
print("  TABLE 2: MWQR — Oil → S&P 500")
print("═" * 70)
oil_df = mwqr_result.get_variable('Oil')
latex2 = wqr.export_latex(
    oil_df,
    value_col='beta',
    pval_col='p_value',
    row_col='quantile',
    col_col='band',
    caption='MWQR: Oil $\\rightarrow$ S\\&P 500',
    label='tab:mwqr_oil',
    digits=4
)
print(latex2)

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  12.2  Causality Results Table
# ══════════════════════════════════════════════════════════════════════

print("═" * 70)
print("  TABLE 3: Nonparametric Quantile Causality — Oil → Gold")
print("═" * 70)

caus_df = caus_mean.to_dataframe()
print(caus_df.to_string(index=False))
print("\nNote: Critical values — 5%: 1.96, 10%: 1.645")

---

## 🎨 13. Additional Visualization Examples

### 13.1  Multi-pair QQ Analysis

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  13.1  Compare QQ surfaces across multiple pairs
# ══════════════════════════════════════════════════════════════════════

pairs = [
    ('Oil',  'SP500', 'Oil → S&P 500'),
    ('Oil',  'Gold',  'Oil → Gold'),
    ('Gold', 'SP500', 'Gold → S&P 500'),
]

fig, axes = plt.subplots(1, 3, figsize=(20, 6), subplot_kw={'projection': '3d'})

for idx, (xvar, yvar, label) in enumerate(pairs):
    res = wqr.qq_regression(
        returns[yvar].values,
        returns[xvar].values,
        verbose=False
    )
    
    mat = res.to_matrix('coefficient')
    y_q = sorted(res.results['y_quantile'].dropna().unique())
    x_q = sorted(res.results['x_quantile'].dropna().unique())
    X, Y = np.meshgrid(x_q, y_q)
    
    from matplotlib.colors import LinearSegmentedColormap
    jet_colors = [
        (0.0, (0.0, 0.0, 0.5)), (0.11, (0.0, 0.0, 1.0)),
        (0.25, (0.0, 0.5, 1.0)), (0.36, (0.0, 1.0, 1.0)),
        (0.50, (0.5, 1.0, 0.5)), (0.64, (1.0, 1.0, 0.0)),
        (0.75, (1.0, 0.5, 0.0)), (0.89, (1.0, 0.0, 0.0)),
        (1.0, (0.5, 0.0, 0.0)),
    ]
    cmap = LinearSegmentedColormap.from_list('jet', [(v,c) for v,c in jet_colors])
    
    ax = axes[idx]
    ax.plot_surface(X, Y, mat, cmap=cmap, alpha=0.9, edgecolor='gray', linewidth=0.2)
    ax.set_xlabel('τ', fontsize=10, labelpad=8)
    ax.set_ylabel('θ', fontsize=10, labelpad=8)
    ax.set_zlabel('β̂', fontsize=10, labelpad=5)
    ax.set_title(label, fontsize=12, fontweight='bold', pad=10)
    ax.view_init(elev=25, azim=-135)

fig.suptitle('Quantile-on-Quantile Regression: Multi-Pair Comparison',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  13.2  WQR across multiple pairs
# ══════════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

wqr_pairs = [
    ('Oil',  'Gold',  'Oil → Gold'),
    ('Oil',  'SP500', 'Oil → S&P 500'),
    ('USD',  'Gold',  'USD → Gold'),
]

for idx, (xvar, yvar, label) in enumerate(wqr_pairs):
    res = wqr.wavelet_qr(
        returns[yvar].values,
        returns[xvar].values,
        wavelet='la8', J=5, bands=True, verbose=False
    )
    
    piv = res.coefficients.pivot(index='level', columns='quantile', values='beta')
    piv_p = res.coefficients.pivot(index='level', columns='quantile', values='p_value')
    
    band_order = ['Short', 'Medium', 'Long']
    ordered = [b for b in band_order if b in piv.index]
    piv = piv.reindex(ordered)
    piv_p = piv_p.reindex(ordered)
    
    ax = axes[idx]
    cmap = LinearSegmentedColormap.from_list('gor',
        [(0, '#8FBC8F'), (0.5, '#FFA500'), (1.0, '#FF0000')])
    im = ax.imshow(piv.values.astype(float), cmap=cmap, aspect='auto')
    ax.set_xticks(range(len(piv.columns)))
    ax.set_xticklabels([f'{q:.2f}' for q in piv.columns], rotation=45, ha='right', fontsize=7)
    ax.set_yticks(range(len(piv.index)))
    ax.set_yticklabels(piv.index)
    ax.set_title(label, fontweight='bold', fontsize=12)
    
    # Stars
    pmat = piv_p.values.astype(float)
    for i in range(pmat.shape[0]):
        for j in range(pmat.shape[1]):
            p = pmat[i,j]
            if not np.isnan(p):
                star = '***' if p < 0.01 else ('**' if p < 0.05 else ('*' if p < 0.10 else ''))
                if star:
                    ax.text(j, i, star, ha='center', va='center', fontsize=8, fontweight='bold')
    
    plt.colorbar(im, ax=ax, shrink=0.8)

fig.suptitle('Wavelet Quantile Regression: Multi-Pair Comparison',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

---

## 📚 14. Summary & Conclusions

### Key Findings

This tutorial demonstrated the complete `wqr` library through an empirical analysis of **Oil–Gold–S&P 500–USD** relationships:

1. **QQ Regression** revealed that the oil–stock dependence is **asymmetric** — strongest during extreme market conditions (lower quantiles) and weakest during moderate conditions.

2. **WQR** showed that the oil–gold relationship varies significantly across **time horizons**, with different effects at short-run vs. long-run frequencies.

3. **MWQR** extended the analysis to a multivariate setting, revealing the **joint effects** of oil, gold, and USD on stock returns at different quantiles and frequencies.

4. **WQQR** provided a refined view of the long-run dependence structure with **pointwise p-values**, revealing which coefficient cells are statistically significant.

5. **Quantile Causality** tests showed evidence of Granger causality from oil to gold in both **mean and variance** at specific quantiles.

6. **WNQC** extended causality testing to wavelet scales, revealing that short-run speculative effects differ from long-run fundamental transmission.

7. **Mediation/Moderation** analysis examined whether gold mediates the oil–stock relationship and whether the interaction effect (moderation) is significant.

8. **WQC** quantified the quantile-specific correlations at each wavelet detail level with bootstrap CIs.

9. **Quantile Density** estimation provided nonparametric estimates of the quantile density function of oil returns.

### Visualization Summary

| Plot Type | Function | Use Case |
|-----------|----------|----------|
| 3D Surface | `plot_qq_3d()`, `plot_wqqr_surface()` | Coefficient landscapes |
| Heatmap | `plot_qq_heatmap()`, `plot_wqr_heatmap()` | Quantile×Band coefficients |
| Contour | `plot_qq_contour()` | Filled isolines |
| Causality | `plot_causality()` | Test stat vs critical values |
| Correlation | `plot_correlation_heatmap()`, `plot_wqc_heatmap()` | Quantile dependence |
| Significance | `plot_wqqr_pvalue_heatmap()` | Binary significance map |
| Mediation | `plot_mediation_panel()` | 5-panel mediation/moderation |
| Density | `plot_quantile_density()` | 4-panel wavelet estimation |

---

## 📖 References

1. **Sim, N. & Zhou, H. (2015).** *Oil prices, US stock return, and the dependence between their quantiles.* Journal of Banking & Finance, 55, 1–12.

2. **Balcilar, M., Gupta, R. & Pierdzioch, C. (2016).** *Does uncertainty move the gold price?* Resources Policy, 49, 74–80.

3. **Balcilar, M., Bekiros, S. & Gupta, R. (2016).** *The role of news-based uncertainty indices in predicting oil markets.* Open Economies Review, 27(2), 229–250.

4. **Song, K., Whang, Y.-J. & Shin, Y. (2012).** *Testing distributional treatment effects.* Econometric Reviews.

5. **Adebayo, T. S. & Ozkan, O. (2023).** *Investigating the influence of socioeconomic conditions on the environment.* Journal of Cleaner Production.

6. **Adebayo, T. S. et al. (2025).** *Unpacking policy ambiguities in residential and commercial renewable energy adoption.* Applied Economics.

7. **Adebayo, T. S., Özkan, O. et al. (2025).** *Environmental Sciences Europe.*

8. **Adebayo, T. S. (2025).** *Can ESG Uncertainty Alter the Emissions Impact of Renewable Energy Consumption?* Statistical Journal of the IAOS.

9. **Chesneau, C., Dewan, I. & Doosti, H.** *Nonparametric estimation of the quantile density function by wavelet methods.*

10. **Roudane, M. (2024).** *QuantileOnQuantile: R package.* CRAN.

11. **Roudane, M. (2024).** *wqc: Wavelet Quantile Correlation.* CRAN.

---

## 📬 Contact

**Dr. Merwan Roudane**  
📧 merwanroudane920@gmail.com  
🔗 [GitHub](https://github.com/merwanroudane/wqrr)  
📦 [PyPI](https://pypi.org/project/wqr/)

---

*© 2025 Dr. Merwan Roudane. MIT License.*